# 3. 其它内置中间件
这里为大部分中间件提供测试代码和输出。

## 3.1 ModelCallLimitMiddleware中间件

限制模型调用次数，避免无限循环，控制调用成本。

参数:
```python
ModelCallLimitMiddleware(
  self,
  *,
  thread_limit: int | None = None,
  run_limit: int | None = None,
  exit_behavior: Literal['end', 'error'] = 'end'
)
```


参数说明：

1. thread_limit：同一个会话（同一个 `thread_id`）里，模型调用的累计次数上限。计数会随 state 保存到 checkpointer，所以需要配合 checkpointer 使用；不配 checkpointer 时，每次 `invoke` 都会重新计数，效果和 run_limit 一样。`None` 表示不限制。
2. run_limit：单次 `invoke` 里模型调用次数的上限，每次 `invoke` 都从 0 重新计数。`None` 表示不限制。它和 thread_limit 的区别见下方“举例3”。
3. exit_behavior：达到上限后怎么处理。
    * `end`（默认）：不再调用模型，直接追加一条内容为 `Model call limits exceeded: ...` 的 AIMessage，结束本次运行；
    * `error`：抛出 `ModelCallLimitExceededError` 异常。

thread_limit 和 run_limit 至少要设置一个，否则创建中间件时会报 `ValueError`。

举例1：整个会话限制-优雅退出

In [20]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool
from pydantic import BaseModel, Field

load_dotenv(override=True, verbose=True, dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

custom_profile = {
    "max_input_tokens": 128_000
}

model = ChatOpenAI(
    model="gpt-5.6-terra",
    api_key=os.environ["OPENAI_API_KEY"],
    base_url=os.environ["OPENAI_API_BASE"],
    #profile=custom_profile
)

In [21]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),  # Required for thread limiting
    tools=[],
    middleware=[
        ModelCallLimitMiddleware(
            thread_limit=2,  # 每个线程最多2次模型调用
            # run_limit=5,   # 每次agent运行最多5次调用模型
            exit_behavior="end",  # 达到限制后退出，还有一个取值是error
        ),
    ],
)
# 保证是同一个会话，同一个线程
config = {"configurable": {"thread_id": "1"}}

print("=" * 30, "> first <", "=" * 30)
response_first = agent.invoke({
    "messages": [HumanMessage("你好")]},
    config=config
)

for msg in response_first["messages"]:
    msg.pretty_print()

print("=" * 30, "> second <", "=" * 30)
response_second = agent.invoke({
    "messages": [HumanMessage("你是谁？")]},
    config=config
)

for msg in response_second["messages"]:
    msg.pretty_print()

print("=" * 30, "> third <", "=" * 30)
response_third = agent.invoke({
    "messages": [HumanMessage("你能帮我做什么？")]},
    config=config
)
for msg in response_third["messages"]:
    msg.pretty_print()

============================== > first < ==============================
================================ Human Message =================================

你好
================================== Ai Message ==================================

 你好！请告诉我需要处理什么代码或工程问题。
============================== > second < ==============================
================================ Human Message =================================

你好
================================== Ai Message ==================================

 你好！请告诉我需要处理什么代码或工程问题。
================================ Human Message =================================

你是谁？
================================== Ai Message ==================================

 我是 Codex，一个面向软件开发的 AI 助手。我可以阅读和修改代码、运行测试、排查问题，并协助完成项目开发。
============================== > third < ==============================
================================ Human Message =================================

你好
================================== Ai Message ==================================

 你好！请告诉我需

分析：

* 本例没有工具，每次 `invoke` 只调用 1 次模型。前两次 `invoke` 后，这个会话累计调用了 2 次。
* 第三次 `invoke` 时累计次数已经达到 `thread_limit=2`，模型没有被调用，中间件直接追加了一条 AIMessage：`Model call limits exceeded: thread limit (2/2)`。

举例2:整个会话限制-抛异常

In [22]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

from typing import List

agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),  # Required for thread limiting
    tools=[],
    middleware=[
        ModelCallLimitMiddleware(
            thread_limit=2,
            # run_limit=5,
            exit_behavior="error",
        ),
    ],
)

config = {"configurable": {"thread_id": "1"}}

print("=" * 30, "> first <", "=" * 30)
response_first = agent.invoke({
    "messages": [HumanMessage("你好")]},
    config=config
)

for msg in response_first["messages"]:
    msg.pretty_print()

print("=" * 30, "> second <", "=" * 30)
response_second = agent.invoke({
    "messages": [HumanMessage("你是谁？")]},
    config=config
)

for msg in response_second["messages"]:
    msg.pretty_print()

print("=" * 30, "> third <", "=" * 30)
response_third = agent.invoke({
    "messages": [HumanMessage("你能帮我做什么？")]},
    config=config
)
for msg in response_third["messages"]:
    msg.pretty_print()

============================== > first < ==============================
================================ Human Message =================================

你好
================================== Ai Message ==================================

你好，有什么需要我帮你处理？
============================== > second < ==============================
================================ Human Message =================================

你好
================================== Ai Message ==================================

你好，有什么需要我帮你处理？
================================ Human Message =================================

你是谁？
================================== Ai Message ==================================

 我是 Codex，一个可以协助你阅读、编写、修改和调试代码的 AI 工程助手。
============================== > third < ==============================


ModelCallLimitExceededError: Model call limits exceeded: thread limit (2/2)

分析：`exit_behavior="error"` 时，第三次 `invoke` 不再追加提示消息，而是直接抛出 `ModelCallLimitExceededError`，需要调用方自己捕获处理。

举例3：run_limit 与 thread_limit 的区别

先弄清两个词：

* **一次运行（run）**：调用一次 `agent.invoke(...)`。Agent 会在“调用模型 → 执行工具 → 再调用模型……”之间循环，直到模型不再调用工具为止，所以**一次运行里模型可能被调用很多次**。
* **一个会话（thread）**：使用同一个 `thread_id` 的多次 `invoke`，例如同一个用户接连问的几个问题。

`run_limit` 限制的是**一次运行**里调用模型的次数，用来防止某一个问题陷入“调用工具 → 再调用模型”的死循环；`thread_limit` 限制的是**整个会话**累计调用的次数，用来控制一个会话的总成本。

两个计数都存放在 state 里（`run_model_call_count` 和 `thread_model_call_count`）。区别在于：`thread_model_call_count` 会随 state 一起保存到 checkpointer，下次 `invoke` 时读出来接着累加；`run_model_call_count` 被声明为“不保存”，每次 `invoke` 都从 0 开始。

真实模型很难稳定地陷入死循环，所以下面用一个假模型来模拟：它每次都要求“再搜一次”，永远不会结束。在同一个会话里连续 `invoke` 两次，对比两种限制：

In [ ]:
import itertools
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from langchain_core.messages import AIMessage, HumanMessage
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver


class LoopingFakeModel(GenericFakeChatModel):
    """假模型：每次都要求再调用一次 search 工具，永不结束，模拟陷入循环的模型。"""

    def bind_tools(self, tools, **kwargs):
        return self  # create_agent 会调用 bind_tools，假模型直接返回自身

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        model_calls["n"] += 1  # 统计模型真正被调用的次数
        return super()._generate(messages, stop=stop, run_manager=run_manager, **kwargs)


def endless_search_calls():
    for i in itertools.count(1):
        yield AIMessage("", tool_calls=[{"name": "search", "args": {"query": f"第{i}次搜索"}, "id": f"call_{i}"}])


@tool
def search(query: str) -> str:
    """搜索资料"""
    return f"{query}：没有找到想要的结果"


model_calls = {"n": 0}


def run_twice(limit_middleware):
    loop_agent = create_agent(
        model=LoopingFakeModel(messages=endless_search_calls()),
        tools=[search],
        checkpointer=InMemorySaver(),
        middleware=[limit_middleware],
    )
    loop_config = {"configurable": {"thread_id": "1"}}  # 两次 invoke 使用同一个会话
    for i in (1, 2):
        model_calls["n"] = 0
        result = loop_agent.invoke({"messages": [HumanMessage(f"第{i}个问题")]}, loop_config)
        print(f"  第 {i} 次 invoke：模型被调用 {model_calls['n']} 次，最后一条消息：{result['messages'][-1].content}")


print("① run_limit=3：每次 invoke 最多调用 3 次模型")
run_twice(ModelCallLimitMiddleware(run_limit=3, exit_behavior="end"))

print("\n② thread_limit=3：同一个会话累计最多调用 3 次模型")
run_twice(ModelCallLimitMiddleware(thread_limit=3, exit_behavior="end"))

① run_limit=3：每次 invoke 最多调用 3 次模型
  第 1 次 invoke：模型被调用 3 次，最后一条消息：Model call limits exceeded: run limit (3/3)
  第 2 次 invoke：模型被调用 3 次，最后一条消息：Model call limits exceeded: run limit (3/3)

② thread_limit=3：同一个会话累计最多调用 3 次模型
  第 1 次 invoke：模型被调用 3 次，最后一条消息：Model call limits exceeded: thread limit (3/3)
  第 2 次 invoke：模型被调用 0 次，最后一条消息：Model call limits exceeded: thread limit (3/3)


分析：

* run_limit=3：两次 `invoke` 都调用了 3 次模型。第 2 次 `invoke` 开始时，计数重新从 0 算起。
* thread_limit=3：第 1 次 `invoke` 就用完了 3 次额度；第 2 次 `invoke` 开始时累计次数已经是 3，模型一次都没有被调用，中间件直接追加了提示消息。
* 实际使用时两者可以同时设置，例如 `run_limit=10, thread_limit=50`：单个问题最多调用 10 次模型，整个会话最多调用 50 次。

## 3.2 ToolCallLimitMiddleware中间件
限制工具调用次数，可以 `限制所有工具` 调用的总次数，也可以 `限制特定工具` 的调用次数。

**作用如下**：
* 避免过多调用某些昂贵的外部API
* 限制网络爬虫或数据库查询请求的数量
* 避免Agent陷入无限循环

**退出行为有三种模式**：
1. error：直接抛异常
2. end：立即结束本次运行（本次 `invoke`），并追加一条说明原因的 AIMessage；同一条 AIMessage 中其他还没执行的工具调用也不再执行
3. continue：继续运行Agent，这是默认行为，此时Agent会将工具调用超出限制的信息传递给模型，后者自主决定后续行为，如果模型能力不足，可能导致死循环.

```python
ToolCallLimitMiddleware(
  self,
  *,
  tool_name: str | None = None,
  thread_limit: int | None = None,
  run_limit: int | None = None,
  exit_behavior: ExitBehavior = 'continue'
)
```

1. tool_name: 限制的具体工具名称。如果为None，则限制适用于所有工具。想分别限制多个工具时，可以添加多个本中间件，每个指定不同的 tool_name。
2. thread_limit: 同一个会话（同一个 `thread_id`）里累计允许的最大工具调用次数，需要配合 checkpointer 使用。None表示无限制
3. run_limit: 单次 `invoke` 里允许的最大工具调用次数，每次 `invoke` 重新计数。None表示无限制。不能大于 thread_limit，否则创建中间件时报 `ValueError`
4. exit_behavior: 超限后的处理方式，取值为 `continue`（默认）、`end`、`error`，含义见上文

注意计数的单位是**工具调用**，不是模型调用：模型在一条 AIMessage 里同时发出 2 个工具调用，就计 2 次。run_limit 与 thread_limit 的区别和 3.1 的举例3 相同。

### 举例1：run_limit 与 thread_limit 的区别

和 3.1 举例3 一样，用一个“每次都要求再搜一次”的假模型模拟死循环，在同一个会话里连续 `invoke` 两次；最后再演示默认的 `continue` 行为。

In [ ]:
import itertools
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from langchain_core.messages import AIMessage, HumanMessage
from langchain.agents import create_agent
from langchain.agents.middleware import ToolCallLimitMiddleware
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver


class FakeToolModel(GenericFakeChatModel):
    """假模型：按顺序返回预先写好的消息，不调用任何接口。"""

    def bind_tools(self, tools, **kwargs):
        return self


def endless_search_calls():
    """模型每次都要求再搜一次，永不结束。"""
    for i in itertools.count(1):
        yield AIMessage("", tool_calls=[{"name": "search", "args": {"query": f"第{i}次搜索"}, "id": f"call_{i}"}])


tool_runs = {"n": 0}


@tool
def search(query: str) -> str:
    """搜索资料"""
    tool_runs["n"] += 1  # 统计工具真正执行的次数
    return f"{query}：没有找到想要的结果"


def run_twice(limit_middleware):
    limit_agent = create_agent(
        model=FakeToolModel(messages=endless_search_calls()),
        tools=[search],
        checkpointer=InMemorySaver(),
        middleware=[limit_middleware],
    )
    limit_config = {"configurable": {"thread_id": "1"}}  # 两次 invoke 使用同一个会话
    for i in (1, 2):
        tool_runs["n"] = 0
        result = limit_agent.invoke({"messages": [HumanMessage(f"第{i}个问题")]}, limit_config)
        print(f"  第 {i} 次 invoke：search 执行 {tool_runs['n']} 次，最后一条消息：{result['messages'][-1].content}")


print("① run_limit=2：每次 invoke 最多执行 2 次工具")
run_twice(ToolCallLimitMiddleware(run_limit=2, exit_behavior="end"))

print("\n② thread_limit=2：同一个会话累计最多执行 2 次工具")
run_twice(ToolCallLimitMiddleware(thread_limit=2, exit_behavior="end"))

print("\n③ run_limit=2 + exit_behavior='continue'（默认值）：超限的调用被拦下，模型继续运行")
continue_agent = create_agent(
    model=FakeToolModel(messages=iter([
        AIMessage("", tool_calls=[{"name": "search", "args": {"query": "第1次搜索"}, "id": "c1"}]),
        AIMessage("", tool_calls=[{"name": "search", "args": {"query": "第2次搜索"}, "id": "c2"}]),
        AIMessage("", tool_calls=[{"name": "search", "args": {"query": "第3次搜索"}, "id": "c3"}]),
        AIMessage("搜索次数已经用完，我根据已有结果回答：……"),
    ])),
    tools=[search],
    middleware=[ToolCallLimitMiddleware(run_limit=2)],
)
tool_runs["n"] = 0
continue_result = continue_agent.invoke({"messages": [HumanMessage("帮我查资料")]})
print(f"  search 执行 {tool_runs['n']} 次")
for msg in continue_result["messages"]:
    if msg.type == "tool":
        print(f"  ToolMessage[{msg.tool_call_id}]：{msg.content}")
print("  最后一条消息：", continue_result["messages"][-1].content)

① run_limit=2：每次 invoke 最多执行 2 次工具
  第 1 次 invoke：search 执行 2 次，最后一条消息：Tool call limit reached: run limit exceeded (3/2 calls).
  第 2 次 invoke：search 执行 2 次，最后一条消息：Tool call limit reached: run limit exceeded (3/2 calls).

② thread_limit=2：同一个会话累计最多执行 2 次工具
  第 1 次 invoke：search 执行 2 次，最后一条消息：Tool call limit reached: thread limit exceeded (3/2 calls).
  第 2 次 invoke：search 执行 0 次，最后一条消息：Tool call limit reached: thread limit exceeded (3/2 calls).

③ run_limit=2 + exit_behavior='continue'（默认值）：超限的调用被拦下，模型继续运行
  search 执行 2 次
  ToolMessage[c1]：第1次搜索：没有找到想要的结果
  ToolMessage[c2]：第2次搜索：没有找到想要的结果
  ToolMessage[c3]：Tool call limit exceeded. Do not make additional tool calls.
  最后一条消息： 搜索次数已经用完，我根据已有结果回答：……


分析：

* ① 和 ② 的区别与 3.1 举例3 相同：run_limit 每次 `invoke` 重新计数，thread_limit 在整个会话里累加。提示中的 `3/2` 表示这次尝试了第 3 次调用（被拦下），上限是 2。
* ③ 使用默认的 `continue`：前 2 次搜索正常执行；第 3 次被拦下，工具没有执行，而是返回一条错误 ToolMessage（“Tool call limit exceeded. Do not make additional tool calls.”），由模型自己决定下一步，这里它直接给出了回答。如果模型不理会这条提示、继续调用工具，就会一直收到错误，可能陷入循环，这时可以再加上 3.1 的 ModelCallLimitMiddleware 兜底。

## 3.3 ModelFallbackMiddleware中间件
用于故障转移，当主模型无法访问时，启用备用模型。

```python
ModelFallbackMiddleware(
  self,
  first_model: str | BaseChatModel,
  *additional_models: str | BaseChatModel = ()
)
```
* additional_models 本身不是一个普通的常规变量名，而是可变位置参数的“容器名”，所以当你尝试用 func(additional_models=...) 赋值时，Python 会认为你正在传递一个名为 additional_models 的普通关键字参数，但函数定义中并没有这个名字的常规形参，从而抛出 TypeError: func() got an unexpected keyword argument。



参数说明：

1. first_model：第一个**备用**模型，可以是模型名称字符串（如 `"openai:gpt-5.5"`，内部用 `init_chat_model` 创建）或模型对象。注意它不是主模型，主模型是 `create_agent` 的 `model` 参数。
2. *additional_models：更多备用模型，按传入顺序依次尝试。

工作方式（`wrap_model_call` 钩子）：先调用主模型；如果抛出异常（模型不存在、网络错误、限流等），就按顺序换用备用模型重试；全部失败时，抛出最后一个异常。

举例1：

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool
from pydantic import BaseModel, Field
from langchain_anthropic import ChatAnthropic
load_dotenv(override=True, verbose=True, dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

custom_profile = {
    "max_input_tokens": 128_000
}

model1 = ChatOpenAI(
    model="gpt-5.6-terra",
    api_key=os.environ["OPENAI_API_KEY"],
    base_url=os.environ["OPENAI_API_BASE"],
    #profile=custom_profile
)

model2 = ChatAnthropic(
    model_name="claude-sonnet-4-6",
    api_key=os.environ["ANTHROPIC_API_KEY"],
    base_url=os.environ["ANTHROPIC_API_BASE"],
    #profile=custom_profile
)

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelFallbackMiddleware
from langchain.messages import HumanMessage

agent = create_agent(
    model="deepseek:fake_model",
    tools=[],
    middleware=[
        ModelFallbackMiddleware(
            model1,
            model2,
        ),
    ],
)

response = agent.invoke({
    "messages": [HumanMessage("你是谁？")]
})

last_msg = response["messages"][-1]
print(last_msg)

print('=' * 30, '-> model_name <-', '=' * 30)

content='我是 Codex，一个基于 GPT-5 的编程助手。我可以帮你阅读和修改代码、运行测试、排查问题，以及实现功能。' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 4391, 'total_tokens': 4427, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 3840, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-terra', 'system_fingerprint': None, 'id': 'resp_0ef7a48813c2b6c1016ac4648e63dc87d2bb802b077d11a8d1', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a10f26-3285-7521-bb95-2cf780cfc489-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 4391, 'output_tokens': 36, 'total_tokens': 4427, 'input_token_details': {'cache_read': 3840}, 'output_token_details': {}}
============================== -> model_name <- ==============================


分析：

* 主模型 `"deepseek:fake_model"` 是一个不存在的模型名，调用失败，于是中间件换用第一个备用模型 `model1`。
* 打印的消息中，`response_metadata` 里的 `'model_name': 'gpt-5.6-terra'` 说明回答来自 `model1`；它调用成功了，所以 `model2` 没有被用到。
* 想直接看到是哪个模型回答的，可以在最后加一行 `print(last_msg.response_metadata["model_name"])`。

## 3.4 LLMToolSelectorMiddleware中间件

智能工具筛选。

当工具太多时，用子模型筛选最相关的几个工具。

```python
LLMToolSelectorMiddleware(
  self,
  *,
  model: str | BaseChatModel | None = None,
  system_prompt: str = DEFAULT_SYSTEM_PROMPT,
  max_tools: int | None = None,
  always_include: list[str] | None = None,
  max_retries: int = 1,
  on_parsing_failure: OnParsingFailure = 'error'
)
```

参数说明：

1. model：负责筛选工具的模型，可以是模型名称字符串或模型对象。不传时使用 Agent 的主模型。通常选一个便宜、快速的模型。
2. system_prompt：给筛选模型的提示词，默认是 “Your goal is to select the most relevant tools for answering the user's query.”。
3. max_tools：最多保留几个筛选出的工具。筛选模型选多了，只保留排在前面的几个（设置后还会在提示词里要求它按相关程度排序）。`None` 表示不限制。
4. always_include：无论筛选结果如何都保留的工具名列表，不占 max_tools 的名额。列表中的工具必须是 Agent 拥有的工具，否则调用时报 `ValueError`。
5. max_retries：筛选模型返回的结果格式不对时，最多重试几次，默认 1 次。
6. on_parsing_failure：重试后格式仍然不对时怎么办：`'error'`（默认，抛出 `ValueError`）、`'none'`（不给主模型任何工具）、`'all'`（给主模型全部工具）、一个工具名列表，或者一个根据结果返回工具名列表的函数。

工作方式（`wrap_model_call` 钩子，每次调用主模型之前执行）：

1. 把 Agent 的工具（always_include 中的除外）作为候选，要求筛选模型按结构化格式 `{"tools": [工具名, ...]}` 返回；
2. 把 system_prompt 和**最后一条用户消息**发给筛选模型，让它按相关程度挑选工具；
3. 按 max_tools 截断，再加上 always_include 中的工具；
4. 只把这些工具交给主模型，其余工具在这次调用中对主模型不可见。

需要注意：

* 筛选只看最后一条用户消息，不看之前的对话；
* 每次调用主模型之前都会筛选一次：一次 `invoke` 中主模型被调用几次，筛选模型就被调用几次，所以筛选模型最好用便宜的；
* 筛选模型选了不存在的工具名时，会报 `ValueError`。

举例1：

为了看清筛选过程，下面用两个假模型：筛选模型返回预先写好的结果（真实场景中由它根据问题和工具说明来选），主模型打印它这次能看到哪些工具。Agent 一共有 6 个工具，`max_tools=2`，`always_include=["get_time"]`。

In [ ]:
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from langchain_core.messages import AIMessage, HumanMessage
from langchain.agents import create_agent
from langchain.agents.middleware import LLMToolSelectorMiddleware
from langchain.tools import tool


class SelectorFakeModel(GenericFakeChatModel):
    """假的筛选模型：打印它收到的消息，再返回预先写好的筛选结果。"""

    def bind_tools(self, tools, **kwargs):
        return self

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        user_msgs = [m.content for m in messages if m.type == "human"]
        print(f"  [筛选模型] 收到 {len(messages)} 条消息，其中用户消息：{user_msgs}")
        return super()._generate(messages, stop=stop, run_manager=run_manager, **kwargs)


class MainFakeModel(GenericFakeChatModel):
    """假的主模型：打印它这次能看到哪些工具。"""

    def bind_tools(self, tools, **kwargs):
        print(f"  [主模型] 能看到的工具：{[t.name for t in tools]}")
        return self


@tool
def get_weather(city: str) -> str:
    """查询指定城市的天气"""
    return f"{city}今天晴"


@tool
def get_news() -> str:
    """查询今日新闻"""
    return "今日无重大新闻"


@tool
def send_email(to: str, body: str) -> str:
    """发送邮件"""
    return "已发送"


@tool
def read_file(path: str) -> str:
    """读取文件内容"""
    return "文件内容"


@tool
def calculator(expression: str) -> str:
    """计算数学表达式"""
    return "42"


@tool
def get_time() -> str:
    """获取当前时间"""
    return "10:00"


# 筛选模型返回的结构化结果：键必须是 tools，值是选中的工具名（按相关程度排序）
selection = {"tools": ["get_weather", "get_news", "calculator"]}
# 返回的结果放在预定义的假模型中
selector_model = SelectorFakeModel(messages=iter([
    AIMessage("", tool_calls=[{"name": "ToolSelectionResponse", "args": selection, "id": f"sel_{i}"}])
    for i in (1, 2)
]))
main_model = MainFakeModel(messages=iter([
    AIMessage("", tool_calls=[{"name": "get_weather", "args": {"city": "北京"}, "id": "call_1"}]),
    AIMessage("北京今天晴。"),
]))

selector_agent = create_agent(
    model=main_model,
    tools=[get_weather, get_news, send_email, read_file, calculator, get_time],
    middleware=[LLMToolSelectorMiddleware(
        model=selector_model,
        max_tools=2,                # 最多保留 2 个筛选出的工具
        always_include=["get_time"],  # 无论筛选结果如何都保留
    )],
)
selector_result = selector_agent.invoke({"messages": [
    HumanMessage("你好"),
    AIMessage("你好！有什么可以帮你？"),
    HumanMessage("北京今天天气怎么样？"),
]})
print("最终回答：", selector_result["messages"][-1].content)

  [筛选模型] 收到 2 条消息，其中用户消息：['北京今天天气怎么样？']
  [主模型] 能看到的工具：['get_weather', 'get_news', 'get_time']
  [筛选模型] 收到 2 条消息，其中用户消息：['北京今天天气怎么样？']
  [主模型] 能看到的工具：['get_weather', 'get_news', 'get_time']
最终回答： 北京今天晴。


分析：

* 筛选模型每次只收到 2 条消息：系统提示词和最后一条用户消息“北京今天天气怎么样？”，前面的“你好”没有发给它。
* 筛选模型选了 3 个工具，`max_tools=2` 只保留了前 2 个（`get_weather`、`get_news`），再加上 `always_include` 中的 `get_time`。主模型只能看到这 3 个工具，看不到其余 3 个。
* 主模型被调用了 2 次（第一次调用 `get_weather`，第二次给出回答），筛选模型也跟着被调用了 2 次。

## 3.5 ToolRetryMiddleware中间件

基于指数退避算法，设置工具调用失败时的重试策略。
> 指数退避（Exponential Backoff） 的核心思想就是：当某个操作失败（通常是网络请求、API 调用或数据库连接）时，系统不会立刻重试，也不会每次都等待相同的固定时间，而是让每一次重试的延迟时间按指数级增长。

为什么不直接重试？

想象一下，某个热门网站的服务器因为瞬间流量太大（比如抢票或秒杀）崩溃了。如果所有失败的客户端都立刻或每隔1秒就重试一次，这无异于对已经瘫痪的服务器进行了一场持续的 DDoS（分布式拒绝服务）攻击，服务器可能永远也缓不过来。

jitter是为了避免大量工具的重试请求集中在固定的时间点，引入抖动。

假设：按照策略，两次工具调用请求的时间间隔应为10秒，加入抖动后，可能为8.9秒，也可能为10.2秒。

```python
ToolRetryMiddleware(
  self,
  *,
  max_retries: int = 2,
  tools: list[BaseTool | str] | None = None,
  retry_on: RetryOn = default_retry_on,
  on_failure: OnFailure = 'continue',
  backoff_factor: float = 2.0,
  initial_delay: float = 1.0,
  max_delay: float = 60.0,
  jitter: bool = True
)
```

参数说明：

1. max_retries：第一次调用失败后最多再重试几次，默认 2，所以总共最多执行 1 + max_retries 次。
2. tools：只对哪些工具启用重试，可以传工具对象或工具名。默认 `None` 表示对所有工具生效；不在列表中的工具失败时不重试。
3. retry_on：哪些异常需要重试。可以是异常类型的元组，例如 `(TimeoutError, ConnectionError)`；也可以是一个函数 `(异常) -> bool`，返回 True 才重试。默认值 `default_retry_on` 会按 LangChain 模型错误类型自带的“是否可重试”来判断，其他异常一律重试，对工具来说基本相当于所有异常都重试。**不在重试范围内的异常会立即抛出，既不重试，也不交给 on_failure 处理。**
4. on_failure：重试次数用完仍然失败时怎么办：
    * `'continue'`（默认）：不抛异常，而是返回一条 ToolMessage，内容是 `Tool 'xxx' failed after N attempts with 异常类型: 异常信息. Please try again.`，由模型决定下一步（下面例子的输出就是这样）；
    * `'error'`：重新抛出异常，Agent 停止运行；
    * 一个函数 `(异常) -> str`：用它的返回值作为这条 ToolMessage 的内容。
5. backoff_factor：指数退避的倍数，默认 2.0。第 n 次重试前等待 `initial_delay × backoff_factor^(n-1)` 秒；设为 0 表示每次都等待固定的 initial_delay。
6. initial_delay：第 1 次重试前等待的秒数，默认 1.0。
7. max_delay：单次等待时间的上限（秒），默认 60.0，防止指数增长后等待过久。
8. jitter：是否给等待时间加上 ±25% 的随机抖动，默认 True。抖动是在 max_delay 截断之后加的，所以实际等待时间可能略高于 max_delay。

max_retries、backoff_factor、initial_delay、max_delay 都不能为负数，否则创建中间件时会报 `ValueError`。

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool
from pydantic import BaseModel, Field

load_dotenv(override=True, verbose=True, dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

custom_profile = {
    "max_input_tokens": 128_000
}

model = ChatOpenAI(
    model="gpt-5.6-terra",
    api_key=os.environ["OPENAI_API_KEY"],
    base_url=os.environ["OPENAI_API_BASE"],
    #profile=custom_profile
)

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ToolRetryMiddleware
from langchain.messages import HumanMessage
import datetime


def write_times(s):
    """将每次工具调用的时间戳和间隔写入本地文件，方便观察退避策略"""
    with open("call_times_with_jitter.txt", "a", encoding="utf-8") as f:
        f.write(s + "\n")


count = 1
start_time = None


@tool
def get_weather(city: str):
    """查询指定城市天气"""
    global count
    global start_time
    interval = 0
    current_time = datetime.datetime.now()
    if not start_time:
        interval = 0
    else:
        # 计算当前调用与上一次调用之间的时间差（秒）
        interval = (current_time - start_time).total_seconds()
    start_time = current_time
    res_str = f"第 {count} 次调用，当前时间： {start_time}, 和上次调用间隔 {interval} 秒"
    count += 1
    # 记录日志
    write_times(res_str)
    # 故意抛出 TimeoutError，以此触发中间件的重试机制
    raise TimeoutError("Not Implemented")


agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[
        # ToolRetryMiddleware 用于捕获工具执行中的异常并自动重试
        ToolRetryMiddleware(
            max_retries=6,  # 最大重试次数（不包含初始的那次调用，一共最多调 1 + 6 = 7 次）
            backoff_factor=2.0,  # 指数退避因子（每次重试等待时间乘以 2）
            initial_delay=1.0,  # 第一次重试前的初始等待时间（1 秒）
            max_delay=10.0,  # 最大等待延迟上限（防止指数增长无限大，限制在 10 秒）
            jitter=True,  # 开启抖动（在等待时间中加入随机性，防止并发请求时出现“惊群效应”）
            retry_on=(TimeoutError,),  # 仅针对捕获到特定的 TimeoutError 异常时才触发重试
            on_failure="continue"  # 当达到最大重试次数依然失败时，Agent 的行为："continue" 表示将错误信息包装后塞回对话历史，让大模型知道失败了并继续决策
        ),
    ],
)

response = agent.invoke({
    "messages": [HumanMessage("今天北京天气如何？")]
})

# 1. 你的提问 -> 2. AI 决定调用工具 -> 3. 重试失败后的错误反馈 -> 4. AI 最终给出的兜底回复
for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

今天北京天气如何？
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_QH0qWrqMNYjcAOPYxior3G83)
 Call ID: call_QH0qWrqMNYjcAOPYxior3G83
  Args:
    city: 北京
================================= Tool Message =================================
Name: get_weather

Tool 'get_weather' failed after 7 attempts with TimeoutError: Not Implemented. Please try again.
================================== Ai Message ==================================

 天气查询服务暂时不可用，暂时无法获取今天北京的实时天气。


### 3.5.2 无抖动

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ToolRetryMiddleware
from langchain.messages import HumanMessage
import datetime


def write_times(s):
    """将每次工具调用的时间戳和间隔写入本地文件，方便观察退避策略"""
    with open("call_times_with_jitter.txt", "a", encoding="utf-8") as f:
        f.write(s + "\n")


count = 1
start_time = None


@tool
def get_weather(city: str):
    """查询指定城市天气"""
    global count
    global start_time
    interval = 0
    current_time = datetime.datetime.now()
    if not start_time:
        interval = 0
    else:
        # 计算当前调用与上一次调用之间的时间差（秒）
        interval = (current_time - start_time).total_seconds()
    start_time = current_time
    res_str = f"第 {count} 次调用，当前时间： {start_time}, 和上次调用间隔 {interval} 秒"
    count += 1
    # 记录日志
    write_times(res_str)
    # 故意抛出 TimeoutError，以此触发中间件的重试机制
    raise TimeoutError("Not Implemented")


agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[
        # ToolRetryMiddleware 用于捕获工具执行中的异常并自动重试
        ToolRetryMiddleware(
            max_retries=6,  # 最大重试次数（不包含初始的那次调用，一共最多调 1 + 6 = 7 次）
            backoff_factor=2.0,  # 指数退避因子（每次重试等待时间乘以 2）
            initial_delay=1.0,  # 第一次重试前的初始等待时间（1 秒）
            max_delay=10.0,  # 最大等待延迟上限（防止指数增长无限大，限制在 10 秒）
            jitter=False,  # 关闭抖动，意味着重试机制从“随机化的指数退避”退化成了“严格固定的指数退避”
            retry_on=(TimeoutError,),  # 仅针对捕获到特定的 TimeoutError 异常时才触发重试
            on_failure="continue"  # 当达到最大重试次数依然失败时，Agent 的行为："continue" 表示将错误信息包装后塞回对话历史，让大模型知道失败了并继续决策
        ),
    ],
)

response = agent.invoke({
    "messages": [HumanMessage("今天北京天气如何？")]
})

# 1. 你的提问 -> 2. AI 决定调用工具 -> 3. 重试失败后的错误反馈 -> 4. AI 最终给出的兜底回复
for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

今天北京天气如何？
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_tyqJbgXiAlqTgaTd6wPKpfsi)
 Call ID: call_tyqJbgXiAlqTgaTd6wPKpfsi
  Args:
    city: 北京
================================= Tool Message =================================
Name: get_weather

Tool 'get_weather' failed after 7 attempts with TimeoutError: Not Implemented. Please try again.
================================== Ai Message ==================================

 天气查询服务暂时不可用，没能获取北京今天的实时天气。


自动重试失败的工具调用。

1. 第 1 次重试（retry_number=0）: 等待 1.0 * (2.0 ** 0) = 1.0 秒
2. 第 2 次重试（retry_number=1）: 等待 1.0 * (2.0 ** 1) = 2.0 秒
3. 第 3 次重试（retry_number=2）: 等待 1.0 * (2.0 ** 2) = 4.0 秒

也就是说，等待时间以指数方式增长 —— 每失败一次，下次再试之前等待更长时间。

注意：源码中 retry_number 从 0 开始计数，所以第 1 次重试前只等待 initial_delay。这和 call_times_with_jitter.txt 中“无抖动”那组的记录一致：间隔依次约为 1、2、4、8 秒，之后被 max_delay=10 截断为 10 秒。

如果你把 backoff_factor = 0，就意味着不使用指数增长，重试之间始终用固定的 initial_delay。

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ToolRetryMiddleware
from langchain.messages import HumanMessage
import datetime


def write_times(s):
    """将每次工具调用的时间戳和间隔写入本地文件，方便观察退避策略"""
    with open("call_times_with_jitter.txt", "a", encoding="utf-8") as f:
        f.write(s + "\n")


count = 1
start_time = None


@tool
def get_weather(city: str):
    """查询指定城市天气"""
    global count
    global start_time
    interval = 0
    current_time = datetime.datetime.now()
    if not start_time:
        interval = 0
    else:
        # 计算当前调用与上一次调用之间的时间差（秒）
        interval = (current_time - start_time).total_seconds()
    start_time = current_time
    res_str = f"第 {count} 次调用，当前时间： {start_time}, 和上次调用间隔 {interval} 秒"
    count += 1
    # 记录日志
    write_times(res_str)
    # 故意抛出 TimeoutError，以此触发中间件的重试机制
    raise TimeoutError("Not Implemented")


agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[
        # ToolRetryMiddleware 用于捕获工具执行中的异常并自动重试
        ToolRetryMiddleware(
            max_retries=6,  # 最大重试次数（不包含初始的那次调用，一共最多调 1 + 6 = 7 次）
            backoff_factor=0,  # 指数退避因子（每次重试等待时间乘以 2）
            initial_delay=1.0,  # 第一次重试前的初始等待时间（1 秒）
            max_delay=10.0,  # 最大等待延迟上限（防止指数增长无限大，限制在 10 秒）
            jitter=False,  # 关闭抖动，意味着重试机制从“随机化的指数退避”退化成了“严格固定的指数退避”
            retry_on=(TimeoutError,),  # 仅针对捕获到特定的 TimeoutError 异常时才触发重试
            on_failure="continue"  # 当达到最大重试次数依然失败时，Agent 的行为："continue" 表示将错误信息包装后塞回对话历史，让大模型知道失败了并继续决策
        ),
    ],
)

response = agent.invoke({
    "messages": [HumanMessage("今天北京天气如何？")]
})

# 1. 你的提问 -> 2. AI 决定调用工具 -> 3. 重试失败后的错误反馈 -> 4. AI 最终给出的兜底回复
for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

今天北京天气如何？
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_0ZCYqqMtqtRSOFzXhzCfGQQu)
 Call ID: call_0ZCYqqMtqtRSOFzXhzCfGQQu
  Args:
    city: 北京
================================= Tool Message =================================
Name: get_weather

Tool 'get_weather' failed after 7 attempts with TimeoutError: Not Implemented. Please try again.
================================== Ai Message ==================================

 暂时无法获取北京的实时天气，天气服务请求超时。


## 3.6 ModelRetryMiddleware中间件
模型调用失败时重试，策略和工具调用的重试一样，都是基于指数退避算法。

因此，本节案例不再重点观察指数退避算法，而是测试不同的退出模式。

```python
ModelRetryMiddleware(
  self,
  *,
  max_retries: int = 2,
  retry_on: RetryOn = default_retry_on,
  on_failure: OnFailure = 'continue',
  backoff_factor: float = 2.0,
  initial_delay: float = 1.0,
  max_delay: float = 60.0,
  jitter: bool = True
)
```

参数说明：

参数和 ToolRetryMiddleware 基本相同（没有 tools 参数），区别在于失败后返回的是 AIMessage：

1. max_retries：第一次调用失败后最多再重试几次，默认 2。
2. retry_on：哪些异常需要重试，写法同 ToolRetryMiddleware。默认值 `default_retry_on` 对模型很有用：各模型集成会把服务商返回的错误转换成 LangChain 的错误类型（本项目用的 `ChatOpenAI` 就是这样），其中**限流（429）、服务端错误（5xx）、连接失败、超时会重试**；**认证失败（401）、没有权限（403）、请求无效（400）、模型不存在（404）不会重试**，因为再试一次结果也一样。其他异常一律重试。不在重试范围内的异常会立即抛出，不经过 on_failure。
3. on_failure：重试次数用完仍然失败时怎么办：
    * `'continue'`（默认）：不抛异常，而是把一条内容为 `Model call failed after N attempts with 异常类型: 异常信息` 的 AIMessage 当作模型的回答。它没有工具调用，所以 Agent 会就此结束，这条消息就是最终回答；
    * `'error'`：重新抛出原来的异常，由调用方处理；
    * 一个函数 `(异常) -> str`：用它的返回值作为这条 AIMessage 的内容，适合给用户返回友好的提示。
4. backoff_factor、initial_delay、max_delay、jitter：等待时间的计算方式和 ToolRetryMiddleware 完全相同，见 3.5。

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool
from pydantic import BaseModel, Field
from langchain_anthropic import ChatAnthropic
load_dotenv(override=True, verbose=True, dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

custom_profile = {
    "max_input_tokens": 128_000
}

model = ChatOpenAI(
    model="gpt-6-luna",
    api_key=os.environ["OPENAI_API_KEY"],
    base_url=os.environ["OPENAI_API_BASE"],
    #profile=custom_profile
)

举例：用会失败的假模型测试重试和不同的退出模式

真实模型很难按需出错，所以本例不使用上面创建的 `model`，而是用一个假模型 `FlakyFakeModel` 来模拟：前 `fail_times` 次调用都抛出 `TimeoutError`，之后才正常回答。为了让例子运行得快一些，`initial_delay` 设为 0.5 秒并关闭抖动，所以两次重试前分别等待 0.5 秒和 1 秒。

In [ ]:
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from langchain_core.messages import AIMessage, HumanMessage
from langchain.agents import create_agent
from langchain.agents.middleware import ModelRetryMiddleware


class FlakyFakeModel(GenericFakeChatModel):
    """假模型：前 fail_times 次调用都抛出 TimeoutError，之后才正常回答。"""

    fail_times: int = 0

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        attempts["n"] += 1
        if attempts["n"] <= self.fail_times:
            print(f"  第 {attempts['n']} 次调用模型：超时失败")
            raise TimeoutError("模型服务响应超时")
        print(f"  第 {attempts['n']} 次调用模型：成功")
        return super()._generate(messages, stop=stop, run_manager=run_manager, **kwargs)


attempts = {"n": 0}


def try_retry_agent(fail_times, **retry_kwargs):
    attempts["n"] = 0
    retry_agent = create_agent(
        model=FlakyFakeModel(fail_times=fail_times, messages=iter([AIMessage("你好！有什么可以帮你？")])),
        middleware=[ModelRetryMiddleware(
            max_retries=2,     # 失败后最多再试 2 次，总共最多调用 3 次
            initial_delay=0.5,  # 为了演示快一些，第 1 次重试前只等 0.5 秒
            jitter=False,
            **retry_kwargs,
        )],
    )
    try:
        retry_result = retry_agent.invoke({"messages": [HumanMessage("你好")]})
        print("  最后一条消息：", retry_result["messages"][-1].content)
    except Exception as e:
        print(f"  抛出异常 {type(e).__name__}: {e}")


print("① 前 2 次失败、第 3 次成功：重试成功，调用方感觉不到出过错")
try_retry_agent(fail_times=2)

print("\n② 一直失败 + on_failure='continue'（默认）：返回一条说明错误的 AIMessage")
try_retry_agent(fail_times=99, on_failure="continue")

print("\n③ 一直失败 + on_failure='error'：重试用完后抛出原来的异常")
try_retry_agent(fail_times=99, on_failure="error")

print("\n④ 一直失败 + on_failure=函数：用函数的返回值作为 AIMessage 的内容")
try_retry_agent(fail_times=99, on_failure=lambda e: f"抱歉，模型服务暂时不可用（{type(e).__name__}），请稍后再试。")

print("\n⑤ retry_on=(ConnectionError,)：超时不在重试范围内，第 1 次失败就直接抛出")
try_retry_agent(fail_times=99, retry_on=(ConnectionError,))

① 前 2 次失败、第 3 次成功：重试成功，调用方感觉不到出过错
  第 1 次调用模型：超时失败
  第 2 次调用模型：超时失败
  第 3 次调用模型：成功
  最后一条消息： 你好！有什么可以帮你？

② 一直失败 + on_failure='continue'（默认）：返回一条说明错误的 AIMessage
  第 1 次调用模型：超时失败
  第 2 次调用模型：超时失败
  第 3 次调用模型：超时失败
  最后一条消息： Model call failed after 3 attempts with TimeoutError: 模型服务响应超时

③ 一直失败 + on_failure='error'：重试用完后抛出原来的异常
  第 1 次调用模型：超时失败
  第 2 次调用模型：超时失败
  第 3 次调用模型：超时失败
  抛出异常 TimeoutError: 模型服务响应超时

④ 一直失败 + on_failure=函数：用函数的返回值作为 AIMessage 的内容
  第 1 次调用模型：超时失败
  第 2 次调用模型：超时失败
  第 3 次调用模型：超时失败
  最后一条消息： 抱歉，模型服务暂时不可用（TimeoutError），请稍后再试。

⑤ retry_on=(ConnectionError,)：超时不在重试范围内，第 1 次失败就直接抛出
  第 1 次调用模型：超时失败
  抛出异常 TimeoutError: 模型服务响应超时


分析：

* ①：前两次超时后自动重试，第 3 次成功，最终回答正常。对调用方来说，就像没出过错。
* ②：3 次都失败后，默认的 `continue` 没有抛异常，而是把 `Model call failed after 3 attempts ...` 当作模型的回答，Agent 就此结束。
* ③：`error` 在重试用完后抛出原来的 `TimeoutError`，需要调用方自己捕获。
* ④：传入函数时，函数的返回值成为最终回答，适合给用户一个友好的提示。
* ⑤：`retry_on` 只包含 `ConnectionError`，`TimeoutError` 不在重试范围内，所以第一次失败就直接抛出，不会重试，也不会经过 on_failure。

## 3.7 LLMToolEmulator中间件

某些情况下，工具尚未开发完成，我们希望先测试工具调用，可以用LLM tool emulator模拟工具。

```python
LLMToolEmulator(
  self,
  *,
  tools: list[str | BaseTool] | None = None,
  model: str | BaseChatModel | None = None
)
```

参数说明：

1. tools：要模拟的工具，可以传工具名或工具对象。默认 `None` 表示模拟**所有**工具；传空列表表示都不模拟；不在列表中的工具照常真实执行。
2. model：负责模拟的模型，可以是模型名称字符串或模型对象。不传时默认使用 `anthropic:claude-sonnet-4-5-20250929`，需要安装 `langchain-anthropic`；这个隐式默认值已被标记为弃用，建议总是显式传入。传字符串时，内部用 `init_chat_model(model, temperature=1)` 创建，较高的 temperature 让每次模拟的结果有所变化。

工作方式（`wrap_tool_call` 钩子）：拦截需要模拟的工具调用，不执行真正的工具函数，而是把工具名、工具说明和调用参数拼成提示词交给模拟模型，要求它“生成这个工具可能返回的真实结果”，再把生成的内容作为 ToolMessage 返回给主模型。

In [ ]:
model2 = ChatAnthropic(
    model_name="claude-sonnet-4-6",
    api_key=os.environ["ANTHROPIC_API_KEY"],
    base_url=os.environ["ANTHROPIC_API_BASE"]
        .rstrip("/")
        .removesuffix("/v1"),
)

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import LLMToolEmulator
from langchain.messages import HumanMessage


@tool
def get_weather(city: str):
    """查询指定城市天气"""
    return f"{city}今天天气晴朗"




agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[
        LLMToolEmulator(
            model=model2,
        )
    ]
)

response = agent.invoke({
    "messages": [HumanMessage("今天北京天气如何")]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

今天北京天气如何
================================== Ai Message ==================================

[{'arguments': '{"city":"北京"}', 'call_id': 'call_t9bIQkZDzrXU1UeVpVcvpszQ', 'name': 'get_weather', 'type': 'function_call', 'id': 'fc_0039048e3cdf6627016ac4b03a42b087d1be5484638144ab14', 'status': 'completed'}]
Tool Calls:
  get_weather (call_t9bIQkZDzrXU1UeVpVcvpszQ)
 Call ID: call_t9bIQkZDzrXU1UeVpVcvpszQ
  Args:
    city: 北京
================================= Tool Message =================================
Name: get_weather

```json
{
  "city": "北京",
  "date": "2025-01-23",
  "weather": "晴转多云",
  "temperature": {
    "current": -3,
    "high": 2,
    "low": -7,
    "unit": "°C"
  },
  "humidity": 42,
  "wind": {
    "direction": "西北风",
    "speed": "3-4级",
    "speed_kmh": 25
  },
  "air_quality": {
    "aqi": 78,
    "level": "良",
    "pm25": 34
  },
  "visibility": "10公里以上",
  "uv_index": 2,
  "sunrise": "07:32",

分析：

* `get_weather` 的真实实现只会返回“北京今天天气晴朗”，但 ToolMessage 是一段内容丰富的 JSON（温度、湿度、空气质量、未来三天预报……），说明真实函数没有执行，结果是 `model2` 根据工具说明和参数编造的。
* 这些数据（比如日期 2025-01-23）都是编的，只适合在工具还没写好时测试 Agent 的流程，不能用于生产环境。
* 本例没有传 `tools`，所以 Agent 的所有工具都会被模拟。

## 3.8 ContextEditingMiddleware中间件

上下文编辑中间件，该中间件提供了上下文管理的一种方式。

通过更改发送给模型的消息列表来控制成本。

**注意**：不会更改消息列表。因此我们只能通过token用量来推测是否对消息列表进行了裁剪。

```python
ContextEditingMiddleware(
  self,
  *,
  edits: Iterable[ContextEdit] | None = None,
  token_count_method: Literal['approximate', 'model'] = 'approximate',
  token_counter: TokenCounter | None = None
)
```

参数说明：

1. edits：上下文编辑策略的列表，每次调用模型前按顺序执行，默认是 `[ClearToolUsesEdit()]`。它到底是什么，见下面的 3.8.1。
2. token_count_method：统计 token 的方式，用来判断是否达到各策略的触发阈值：
    * `'approximate'`（默认）：按字符数估算，速度快但不精确；
    * `'model'`：调用模型自己的 `get_num_tokens_from_messages` 精确统计（会把系统提示词和工具定义一起算进去），更准确但更慢，而且需要模型支持。
3. token_counter：自定义的 token 计数函数（接收消息列表，返回整数）。传入后优先使用它，token_count_method 会被忽略。例如对话中中文很多、默认的估算不准时，可以换成自己的计数方法。

### 3.8.1 edits 参数是什么

`edits` 是一个列表，里面的每个元素是一个“编辑策略”对象。中间件通过 `wrap_model_call` 钩子，在每次调用模型之前做三件事：

1. 把要发给模型的消息列表**复制**一份；
2. 按列表顺序，依次调用每个策略的 `apply(消息副本, count_tokens=计数函数)`，由策略直接修改这份副本；
3. 把修改后的副本发给模型。

因为修改的是副本，state 中保存的消息不受影响，这就是开头说“不会更改消息列表”的原因。

什么对象能放进 `edits`？只要有一个这样的 `apply` 方法就可以。源码用 `Protocol`（只规定“要有哪些方法”的接口）定义了它：

```python
class ContextEdit(Protocol):
    def apply(self, messages: list[AnyMessage], *, count_tokens: TokenCounter) -> None:
        """直接修改 messages（就地修改，不返回新列表）"""
```

* `messages`：消息列表的副本，策略可以直接替换、删除其中的元素；
* `count_tokens`：中间件按 token_count_method / token_counter 准备好的计数函数，策略用它判断是否需要动手。

**除了 ClearToolUsesEdit 还有其他策略吗？** LangChain（本项目安装的 1.4.2）内置的只有 `ClearToolUsesEdit` 一个，源码注释也写着目前只支持这一个策略，它对应 Anthropic 服务端的 `clear_tool_uses_20250919` 行为。不过 `edits` 只要求对象有 `apply` 方法，所以可以自己写策略，见 3.8.4。

另外，`ChatAnthropic` 和 `ChatOpenAI` 自身也有 `context_management` 参数，那是由模型厂商在服务端完成的上下文管理，和这个中间件无关。

### 3.8.2 ClearToolUsesEdit：清理较早的工具结果

最常用（也是目前唯一内置）的上下文编辑策略:
```python
ClearToolUsesEdit(
  self,
  trigger: int = 100000,
  clear_at_least: int = 0,
  keep: int = 3,
  clear_tool_inputs: bool = False,
  exclude_tools: Sequence[str] = (),
  placeholder: str = DEFAULT_TOOL_PLACEHOLDER
)
```

ClearToolUsesEdit 的参数：

1. trigger：触发阈值（token 数），默认 100000。本次要发给模型的消息 token 总数超过它才会清理，否则什么也不做。
2. clear_at_least：每次至少要腾出多少 token，默认 0。为 0 时，会把所有可以清理的工具结果都清理掉；大于 0 时，从最早的工具结果开始逐条清理，腾出的 token 够了就停止。
3. keep：保留最近的几条工具结果不清理，默认 3。
4. clear_tool_inputs：是否同时清空对应 AIMessage 中这次工具调用的参数（`args` 变成 `{}`），默认 False。
5. exclude_tools：永远不清理的工具名列表，适合结果很关键、后面还会用到的工具。
6. placeholder：被清理的工具结果替换成的文字，默认 `"[cleared]"`。

清理时只替换 ToolMessage 的内容，消息本身和对应的工具调用都还在，所以模型仍然知道“之前调用过这个工具”，只是看不到结果了。

### 3.8.3 举例：观察模型实际收到了什么

开头提到只能通过 token 用量推测是否裁剪了消息。其实用一个会打印输入的假模型就能直接看到：下面的假模型先连续调用 4 次 `search`（每次返回 211 个字的长结果），最后给出回答；中间件设置为超过 200 token 时清理，只保留最近 1 条工具结果。

In [26]:
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from langchain_core.messages import AIMessage, HumanMessage
from langchain.agents import create_agent
from langchain.agents.middleware import ClearToolUsesEdit, ContextEditingMiddleware
from langchain.tools import tool
from collections.abc import Iterator
from pydantic import Field

def show(text):
    """短内容原样显示；长内容只显示开头和总字数。"""
    return text if len(text) <= 20 else f"{text[:9]}…({len(text)}字)"


class PeekFakeModel(GenericFakeChatModel):
    """假模型：打印它实际收到的工具结果（每条只显示开头），再返回预先写好的消息。"""
    # 预设响应供模型逐条消费，不参与追踪序列化
    messages: Iterator[AIMessage | str] = Field(exclude=True)

    def bind_tools(self, tools, **kwargs):
        return self

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        print("  模型收到的工具结果：", [show(m.content) for m in messages if m.type == "tool"])
        return super()._generate(messages, stop=stop, run_manager=run_manager, **kwargs)


@tool
def search(query: str) -> str:
    """搜索资料，返回很长的结果"""
    return f"【{query}】的搜索结果：" + "很长的内容" * 40


def four_searches_then_answer():
    """模型先连续搜索 4 次，最后给出回答。"""
    return iter(
        [AIMessage("", tool_calls=[{"name": "search", "args": {"query": f"问题{i}"}, "id": f"call_{i}"}])
         for i in range(1, 5)]
        + [AIMessage("4 个问题都查完了。")]
    )


edit_agents = create_agent(
    model=PeekFakeModel(messages=four_searches_then_answer()),
    tools=[search],
    middleware=[ContextEditingMiddleware(edits=[
        ClearToolUsesEdit(trigger=200, keep=1),  # 超过 200 token 时清理，只保留最近 1 条工具结果
    ])],
)
edit_result = edit_agents.invoke({"messages": [HumanMessage("帮我查 4 个问题")]})
print("state 中保存的工具结果：", [show(m.content) for m in edit_result["messages"] if m.type == "tool"])

  模型收到的工具结果： []
  模型收到的工具结果： ['【问题1】的搜索结…(211字)']
  模型收到的工具结果： ['【问题1】的搜索结…(211字)', '【问题2】的搜索结…(211字)']
  模型收到的工具结果： ['[cleared]', '[cleared]', '【问题3】的搜索结…(211字)']
  模型收到的工具结果： ['[cleared]', '[cleared]', '[cleared]', '【问题4】的搜索结…(211字)']
state 中保存的工具结果： ['【问题1】的搜索结…(211字)', '【问题2】的搜索结…(211字)', '【问题3】的搜索结…(211字)', '【问题4】的搜索结…(211字)']


In [28]:
from rich import print as rprint
rprint(edit_result)

{
    'messages': [
        HumanMessage(
            content='帮我查 4 个问题',
            additional_kwargs={},
            response_metadata={},
            id='280fb810-0028-4ff7-b881-6eb7311705a3'
        ),
        AIMessage(
            content='',
            additional_kwargs={},
            response_metadata={},
            id='lc_run--01a110b4-f29a-7e72-b468-22cef0c69688-0',
            tool_calls=[{'name': 'search', 'args': {'query': '问题1'}, 'id': 'call_1', 'type': 'tool_call'}],
            invalid_tool_calls=[]
        ),
        ToolMessage(
            content='【问题1】的搜索结果：很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很
长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的
内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容
很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容',
            name='search',
            id='a08d1142-7ac7-4a49-b423-fa2495403d4a',
            tool_call_id='call_1'
        ),
        AIMessage(
            content='',
            additional_kwargs={},
            response_metadata={},
            id='lc_run--01a110b4-f29e-7e30-a119-9bf9155dbe8c-0',
            tool_calls=[{'name': 'search', 'args': {'query': '问题2'}, 'id': 'call_2', 'type': 'tool_call'}],
            invalid_tool_calls=[]
        ),
        ToolMessage(
            content='【问题2】的搜索结果：很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很
长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的
内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容
很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容',
            name='search',
            id='fd0eb355-3eac-427e-853c-74e9a4aa5861',
            tool_call_id='call_2'
        ),
        AIMessage(
            content='',
            additional_kwargs={},
            response_metadata={},
            id='lc_run--01a110b4-f2a0-7700-85c3-e85bca04d642-0',
            tool_calls=[{'name': 'search', 'args': {'query': '问题3'}, 'id': 'call_3', 'type': 'tool_call'}],
            invalid_tool_calls=[]
        ),
        ToolMessage(
            content='【问题3】的搜索结果：很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很
长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的
内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容
很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容',
            name='search',
            id='c93cf9a2-f604-4df0-b241-ff1b6213382d',
            tool_call_id='call_3'
        ),
        AIMessage(
            content='',
            additional_kwargs={},
            response_metadata={},
            id='lc_run--01a110b4-f2a3-7e91-9abe-e2053b5f455f-0',
            tool_calls=[{'name': 'search', 'args': {'query': '问题4'}, 'id': 'call_4', 'type': 'tool_call'}],
            invalid_tool_calls=[]
        ),
        ToolMessage(
            content='【问题4】的搜索结果：很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很
长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的
内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容
很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容',
            name='search',
            id='aba5f267-37eb-4cdd-99e4-10b5c3b80cc0',
            tool_call_id='call_4'
        ),
        AIMessage(
            content='4 个问题都查完了。',
            additional_kwargs={},
            response_metadata={},
            id='lc_run--01a110b4-f2a5-7dd0-8a60-bd5869fd7cc9-0',
            tool_calls=[],
            invalid_tool_calls=[]
        )
    ]
}

分析：

* 前 3 次调用模型时，工具结果最多 2 条，估算的 token 数（约 93、179）没超过 200，模型收到的是完整结果。
* 第 4 次调用时有 3 条结果（约 265 token），超过阈值：较早的 2 条被替换成 `[cleared]`，只保留最近 1 条；第 5 次调用时同样只保留最近 1 条。
* 打印 state 时（上面代码的最后一行，以及 `rprint(edit_result)`），4 条工具结果都完整保留，看不到 `[cleared]`。原因见下。

**为什么 `edit_result` 里没有 `[cleared]`**

`edit_result` 是 Agent 运行结束后的 state，而 ContextEditingMiddleware 从来不修改 state。它在每次调用模型前只做三件事（`langchain/agents/middleware/context_editing.py` 第 270–274 行）：

```python
edited_messages = deepcopy(list(request.messages))              # 复制一份要发给模型的消息
for edit in self.edits:
    edit.apply(edited_messages, count_tokens=count_tokens)      # 策略修改的是这份副本
return handler(request.override(messages=edited_messages))     # 把副本发给模型
```

这份副本只在这一次模型调用中使用。调用结束后，写回 state 的只有模型的回复（AIMessage），副本直接丢弃；下一次调用模型时，再从完整的 state 重新复制、重新清理。可以理解为：模型每次看到的是一份删减过的复印件，原件（state）始终完整。

所以 `[cleared]` 只出现在“模型收到的工具结果”那几行里：它们是假模型在调用过程中打印的，看到的是副本；打印 state 时看到的则是原件。

这正是它和 SummarizationMiddleware 的区别：SummarizationMiddleware 在 `before_model` 中返回 `RemoveMessage(id=REMOVE_ALL_MESSAGES)` 和摘要，直接改写 state，所以 02 笔记的返回结果里能看到摘要消息。ContextEditing 这样设计，是为了保留完整的历史：前端展示、后续的对话轮次和其他中间件用的都是完整版本，只有模型看到的是精简版。

**在 invoke 之后查看模型收到了什么**

如果想在 invoke 结束后再查看模型收到的内容，可以让假模型把每次收到的消息记下来。用真实模型时，也可以在 LangSmith 中打开每次模型调用，查看它的输入。下面的代码使用上面定义的 `PeekFakeModel`、`search` 等，需要先运行上面的代码单元格：

In [ ]:
from rich import print as rprint

received = []  # 记录每次调用模型时实际收到的消息


class RecordingPeekModel(PeekFakeModel):
    """在 PeekFakeModel 的基础上，把每次收到的消息记录到 received 中。"""

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        received.append(messages)
        return super()._generate(messages, stop=stop, run_manager=run_manager, **kwargs)


record_agent = create_agent(
    model=RecordingPeekModel(messages=four_searches_then_answer()),
    tools=[search],
    middleware=[ContextEditingMiddleware(edits=[ClearToolUsesEdit(trigger=200, keep=1)])],
)
record_result = record_agent.invoke({"messages": [HumanMessage("帮我查 4 个问题")]})

print(f"\n模型一共被调用了 {len(received)} 次，下面是最后一次调用时模型收到的消息：")
rprint(received[-1])
print("同一次运行的 state 中保存的工具结果：", [show(m.content) for m in record_result["messages"] if m.type == "tool"])

  模型收到的工具结果： []
  模型收到的工具结果： ['【问题1】的搜索结…(211字)']
  模型收到的工具结果： ['【问题1】的搜索结…(211字)', '【问题2】的搜索结…(211字)']
  模型收到的工具结果： ['[cleared]', '[cleared]', '【问题3】的搜索结…(211字)']
  模型收到的工具结果： ['[cleared]', '[cleared]', '[cleared]', '【问题4】的搜索结…(211字)']

模型一共被调用了 5 次，下面是最后一次调用时模型收到的消息：


[
    HumanMessage(
        content='帮我查 4 个问题',
        additional_kwargs={},
        response_metadata={},
        id='f9fcc375-7623-4d08-8c8e-a05de68ec241'
    ),
    AIMessage(
        content='',
        additional_kwargs={},
        response_metadata={},
        id='lc_run--01a110c1-7f9f-7c71-9d3e-cf723beb5e26-0',
        tool_calls=[{'name': 'search', 'args': {'query': '问题1'}, 'id': 'call_1', 'type': 'tool_call'}],
        invalid_tool_calls=[]
    ),
    ToolMessage(
        content='[cleared]',
        name='search',
        id='69fc3dcb-733f-4a32-ba0b-4789028d1903',
        tool_call_id='call_1'
    ),
    AIMessage(
        content='',
        additional_kwargs={},
        response_metadata={},
        id='lc_run--01a110c1-7fa0-7e41-bb2b-71b89ef1d5d5-0',
        tool_calls=[{'name': 'search', 'args': {'query': '问题2'}, 'id': 'call_2', 'type': 'tool_call'}],
        invalid_tool_calls=[]
    ),
    ToolMessage(
        content='[cleared]',
        name='search',
        id='14593a0e-a5b4-4f1b-b0ad-f612d7f6ce64',
        tool_call_id='call_2'
    ),
    AIMessage(
        content='',
        additional_kwargs={},
        response_metadata={},
        id='lc_run--01a110c1-7fa1-7621-828e-b8a232f302a2-0',
        tool_calls=[{'name': 'search', 'args': {'query': '问题3'}, 'id': 'call_3', 'type': 'tool_call'}],
        invalid_tool_calls=[]
    ),
    ToolMessage(
        content='[cleared]',
        name='search',
        id='9f766e70-cd66-4ed6-8eb9-bc13a12c34fd',
        tool_call_id='call_3'
    ),
    AIMessage(
        content='',
        additional_kwargs={},
        response_metadata={},
        id='lc_run--01a110c1-7fa1-7621-828e-b8efcfbfc156-0',
        tool_calls=[{'name': 'search', 'args': {'query': '问题4'}, 'id': 'call_4', 'type': 'tool_call'}],
        invalid_tool_calls=[]
    ),
    ToolMessage(
        content='【问题4】的搜索结果：很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的
内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容
很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长
的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容很长的内容',
        name='search',
        id='af42ba92-c11f-4179-be5e-3588a6ff6faf',
        tool_call_id='call_4'
    )
]

同一次运行的 state 中保存的工具结果： ['【问题1】的搜索结…(211字)', '【问题2】的搜索结…(211字)', '【问题3】的搜索结…(211字)', '【问题4】的搜索结…(211字)']


分析：

* `received` 记录了 5 次模型调用的输入，`received[-1]` 是最后一次：前 3 条 ToolMessage 的内容都是 `[cleared]`，只有最近 1 条保持完整，这才是模型真正看到的内容。
* 同一次运行的 state（`record_result`）和上面的 `edit_result` 一样，4 条工具结果都完整保留。

### 3.8.4 自定义编辑策略

下面写一个自定义策略 `TruncateToolOutputEdit`：token 数超过 trigger 时，把每条工具结果截断到 max_chars 个字符。它只需要实现 `apply` 方法，不需要继承任何类。

把它和 ClearToolUsesEdit 一起放进 `edits`，观察两个策略按顺序执行的效果。本例使用 3.8.3 中定义的 `PeekFakeModel`、`search` 等，需要先运行上一个代码单元格。

In [29]:
from dataclasses import dataclass
from langchain_core.messages import ToolMessage


@dataclass
class TruncateToolOutputEdit:
    """自定义编辑策略：token 数超过 trigger 时，把每条工具结果截断到 max_chars 个字符。"""

    trigger: int = 0
    max_chars: int = 20

    def apply(self, messages, *, count_tokens):
        if count_tokens(messages) <= self.trigger:  # 没超过阈值就什么也不做
            return
        for i, msg in enumerate(messages):
            if isinstance(msg, ToolMessage) and len(msg.content) > self.max_chars:
                # 就地替换列表中的元素；messages 是中间件复制出来的副本，不会影响 state
                messages[i] = msg.model_copy(update={"content": msg.content[: self.max_chars] + "…(已截断)"})


custom_edit_agent = create_agent(
    model=PeekFakeModel(messages=four_searches_then_answer()),
    tools=[search],
    middleware=[ContextEditingMiddleware(edits=[
        ClearToolUsesEdit(trigger=200, keep=1),           # 第 1 个策略：清理较早的工具结果
        TruncateToolOutputEdit(trigger=100, max_chars=9),  # 第 2 个策略：再截断剩下的长结果
    ])],
)
custom_edit_result = custom_edit_agent.invoke({"messages": [HumanMessage("帮我查 4 个问题")]})
print("state 中保存的工具结果：", [show(m.content) for m in custom_edit_result["messages"] if m.type == "tool"])

  模型收到的工具结果： []
  模型收到的工具结果： ['【问题1】的搜索结…(211字)']
  模型收到的工具结果： ['【问题1】的搜索结…(已截断)', '【问题2】的搜索结…(已截断)']
  模型收到的工具结果： ['[cleared]', '[cleared]', '【问题3】的搜索结…(已截断)']
  模型收到的工具结果： ['[cleared]', '[cleared]', '[cleared]', '【问题4】的搜索结…(已截断)']
state 中保存的工具结果： ['【问题1】的搜索结…(211字)', '【问题2】的搜索结…(211字)', '【问题3】的搜索结…(211字)', '【问题4】的搜索结…(211字)']


分析：

* 第 3 次调用时 token 数约 179：没超过 ClearToolUsesEdit 的 200，但超过了自定义策略的 100，所以 2 条结果都被截断。
* 第 4、5 次调用时，先由 ClearToolUsesEdit 把较早的结果换成 `[cleared]`，再由自定义策略截断剩下的那条。**edits 中的策略按列表顺序执行，后一个策略看到的是前一个策略修改后的结果。**
* state 中的消息仍然完整，自定义策略修改的同样只是副本。

## 3.9 FilesystemFileSearchMiddleware中间件

基于系统的Glob和Grep检索工具，为Agent赋予本地文件搜索和分析的能力。
* Glob根据文件路径检索
* Grep根据文件内容检索

**为什么需要这个中间件？Agent 不是本身就能搜索文件吗？**

不能。**Agent 里的模型本身碰不到任何文件。** 模型能做的只有两件事：输出文字，或者输出“我想调用某个工具”的请求。真正去磁盘上查找、读取文件的是工具函数，也就是在本机运行的 Python 代码。没有提供工具时，模型对磁盘一无所知，只能回答“无法访问”，或者编造一个答案。

05 笔记 TodoList 的例子就是这样：为了让 Agent 修复代码，我们自己写了 `list_files`、`read_file`、`write_file`、`run_tests` 四个工具。Claude Code、Codex 这类编程助手能搜索文件，也是因为它们自带了文件工具。

FilesystemFileSearchMiddleware 的意义，就是提供**现成、安全**的文件搜索工具。它会自动注册两个工具，所以下面例子中的 `tools=[]` 也能搜索文件：

| 工具 | 参数（由模型在调用时填写） | 作用 |
|---|---|---|
| `glob_search` | `pattern`：文件名或路径模式，如 `**/*.py`<br>`path`：从哪个目录开始找，默认 `/` | 按文件名找文件，返回匹配的文件路径，按修改时间从新到旧排列 |
| `grep_search` | `pattern`：正则表达式<br>`path`：同上<br>`include`：按文件名过滤，如 `*.py`、`*.{ts,tsx}`<br>`output_mode`：`files_with_matches`（只返回文件路径，默认）/ `content`（返回 `文件:行号:内容`）/ `count`（返回 `文件:匹配次数`） | 按内容搜索文件 |

和自己手写工具相比，它还做了两件容易被忽略的事：

* **限制在 `root_path` 之内**：模型看到的是以 `/` 开头的虚拟路径（如 `/my_add.py`），看不到真实的绝对路径；含 `..` 或 `~` 的路径会被直接拒绝，指向根目录外的符号链接也会被排除。05 中手写的 `read_file` 就需要自己用 `startswith` 检查是否越界。
* **搜索速度快**：`grep_search` 会优先调用 ripgrep，没有安装时自动改用 Python 实现。

**局限**：它只能“找”，不能“读”和“写”，没有读取整个文件或修改文件的工具。想让 Agent 读写文件，还需要自己写工具，或者使用 ShellToolMiddleware 等能执行操作的中间件。

```python

FilesystemFileSearchMiddleware(
  self,
  *,
  root_path: str,
  use_ripgrep: bool = True,
  max_file_size_mb: int = 10
)
```

参数说明：

1. root_path（必填）：允许搜索的根目录。模型使用的虚拟路径 `/` 就代表这个目录，所有搜索都被限制在其中。相对路径以 Python 进程的当前工作目录为准，在 Jupyter 中通常就是笔记所在的目录，所以例子里写的是 `../todo_workspace`。
2. use_ripgrep：`grep_search` 是否优先使用 ripgrep（命令行工具 `rg`，需要单独安装，例如 `brew install ripgrep`），默认 True。没有安装或调用失败时，会自动改用 Python 实现，不会报错。它只影响 `grep_search`，`glob_search` 始终使用 Python 的 `pathlib`。
3. max_file_size_mb：搜索内容时跳过超过这个大小的文件（单位 MB），默认 10。注意：源码只在 Python 实现中检查文件大小，调用 ripgrep 时不会传入这个限制。所以在装了 ripgrep 且 `use_ripgrep=True` 时，这个参数实际上不起作用。

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import FilesystemFileSearchMiddleware
from langchain.messages import HumanMessage

agent = create_agent(
    model=model,
    tools=[],  # 自动添加 Glob 和 Grep 工具
    middleware=[
        FilesystemFileSearchMiddleware(
            root_path="../todo_workspace",  #搜索目录
            # 是否启用 ripgrep 搜索引擎：
            # 设为 True 可以获得比原生 Grep 更快的性能（前提是系统已安装 ripgrep）
            use_ripgrep=True,
            # 单个文件的最大读取限制（单位MB）：防止读取超大型日志或二进制文件导致 OOM
            max_file_size_mb=10
        ),
    ],
)

result = agent.invoke({
    "messages": [HumanMessage("找到包含add函数的Python或Jupyter文件")]
})

for msg in result["messages"]:
    msg.pretty_print()

/Users/skk/Developer/lang-chain-learn/.venv/lib/python3.11/site-packages/langchain/agents/middleware/file_search.py:237: FutureWarning: Possible nested set at position 4
  re.compile(pattern)
/Users/skk/Developer/lang-chain-learn/.venv/lib/python3.11/site-packages/langchain/agents/middleware/file_search.py:237: FutureWarning: Possible nested set at position 19
  re.compile(pattern)
/Users/skk/Developer/lang-chain-learn/.venv/lib/python3.11/site-packages/langchain/agents/middleware/file_search.py:237: FutureWarning: Possible nested set at position 34
  re.compile(pattern)
/Users/skk/Developer/lang-chain-learn/.venv/lib/python3.11/site-packages/langchain/agents/middleware/file_search.py:237: FutureWarning: Possible nested set at position 9
  re.compile(pattern)
/Users/skk/Developer/lang-chain-learn/.venv/lib/python3.11/site-packages/langchain/agents/middleware/file_search.py:237: FutureWarning: Possible nested set at position 24
  re.compile(pattern)


================================ Human Message =================================

找到包含add函数的Python或Jupyter文件
================================== Ai Message ==================================
Tool Calls:
  grep_search (call_AVKyp6PgERClptwCBfPQf7ys)
 Call ID: call_AVKyp6PgERClptwCBfPQf7ys
  Args:
    pattern: (^|[[:space:]])def[[:space:]]+add[[:space:]]*\(
    path: /
    include: *.py
    output_mode: files_with_matches
  grep_search (call_wSzzeHfQrTU5FDH8DMFvYYfg)
 Call ID: call_wSzzeHfQrTU5FDH8DMFvYYfg
  Args:
    pattern: ["']?def[[:space:]]+add[[:space:]]*\(
    path: /
    include: *.ipynb
    output_mode: files_with_matches
  glob_search (call_wCtuZK56mAIH3XANVzSKJSeU)
 Call ID: call_wCtuZK56mAIH3XANVzSKJSeU
  Args:
    pattern: **/*.{py,ipynb}
    path: /
================================= Tool Message =================================
Name: grep_search

/my_add.py
================================= Tool Message =================================
Name: grep_search

No matches found


分析：

* 模型先后调用了中间件提供的 `grep_search` 和 `glob_search`，在 `/my_add.py`（也就是 `../todo_workspace/my_add.py`）中找到了 `add` 函数。结果中的路径都是以 `/` 开头的虚拟路径。
* `glob_search` 返回 “No files found”，并不是目录里没有文件：它使用 Python 的 `pathlib` glob，不支持模型写的 `{py,ipynb}` 花括号写法，换成 `**/*.py` 就能找到 `my_add.py` 和 `test_my_add.py`。`grep_search` 的 `include` 参数则支持花括号。
* 输出开头的 `FutureWarning`：`grep_search` 会先用 Python 的 `re.compile` 检查正则是否合法，而模型写的 `[[:space:]]` 是 ripgrep 支持、Python 不支持的写法，Python 因此发出警告，但不影响搜索结果。
* 代码注释中的 `allowed_extensions` 不是这个中间件的参数，取消注释会报 `TypeError: unexpected keyword argument`。想限制文件类型，可以在提示词中说明，由模型通过 `grep_search` 的 `include` 参数来过滤。
* 本机装了 ripgrep（`/opt/homebrew/bin/rg`），所以例子中的 `max_file_size_mb=10` 实际上没有生效，原因见参数说明。

# 4. 多个中间件组合及执行顺序
问题：Middleware 可以叠加使用，那么多个中间件书写顺序重要吗？

非常重要！

比如：
```python
middleware=[
    TrimmerMiddleware(), # 1. 先修剪消息
    SummarizationMiddleware(), # 2. 再摘要
    LoggingMiddleware() # 3. 最后记录日志
]
```

下面使用自定义中间件来观察:

In [4]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool
from pydantic import BaseModel, Field

load_dotenv(override=True, verbose=True, dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

custom_profile = {
    "max_input_tokens": 128_000
}

model = ChatOpenAI(
    model="gpt-5.6-terra",
    api_key=os.environ["OPENAI_API_KEY"],
    base_url=os.environ["OPENAI_API_BASE"],
    #profile=custom_profile
)

In [5]:
from langchain.agents.middleware import AgentMiddleware
from langchain.agents import create_agent

class Middleware1(AgentMiddleware):
    def before_model(self, state, runtime):
        print("[中间件1] before_model")
        return None

    def after_model(self, state, runtime):
        print("[中间件1] after_model")
        return None


class Middleware2(AgentMiddleware):
    def before_model(self, state, runtime):
        print("[中间件2] before_model")
        return None

    def after_model(self, state, runtime):
        print("[中间件2] after_model")
        return None


class Middleware3(AgentMiddleware):
    def before_model(self, state, runtime):
        print("[中间件3] before_model")
        return None

    def after_model(self, state, runtime):
        print("[中间件3] after_model")
        return None


agent = create_agent(
    model=model,
    tools=[],
    middleware=[Middleware3(), Middleware1(), Middleware2()]
)

print("\n执行一次调用，观察顺序：")
agent.invoke({"messages": [{"role": "user", "content": "测试"}]})


执行一次调用，观察顺序：
[中间件3] before_model
[中间件1] before_model
[中间件2] before_model
[中间件2] after_model
[中间件1] after_model
[中间件3] after_model


{'messages': [HumanMessage(content='测试', additional_kwargs={}, response_metadata={}, id='ecfe61c6-dc99-4c1b-ae82-0219ac750c63'),
  AIMessage(content=' 测试成功。你好，Linda！', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 3678, 'total_tokens': 3710, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 19, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 3313, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-terra', 'system_fingerprint': None, 'id': 'resp_ft_512b6e7a101eba0bcfbc3d7100a8bd01', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1114a-b6e9-7ad1-b3d5-ea710ae6844e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 3678, 'output_tokens': 32, 'total_tokens': 37